# GATraj: fold 1 (Hotel) on Google Colab
Enable a GPU in **Runtime > Change runtime type**, then run the cells in order.

Training uses one 1,000-epoch schedule and backs up a checkpoint to Drive after each 100-epoch segment. Resume searches **local Colab storage only**. Drive is used only for backups; no backup directories are deleted.

New runs use `colab_savedata` to avoid the pretrained checkpoints bundled in `savedata`. Old checkpoints are preserved, but this notebook does not automatically resume files that previously failed to load. After a runtime reset, manually restore a compatible checkpoint into the printed local model directory before running the training cell.


In [ ]:
from pathlib import Path
import subprocess
import sys

TEST_SET = 1  # Hotel
TOTAL_EPOCHS = 1000
BACKUP_EVERY = 100
SEED = 1
REPO_DIR = Path('/content/GATraj')
REPO_URL = 'https://github.com/Kedar2002/GATraj.git'
LOCAL_SAVE_BASE = REPO_DIR / 'colab_savedata'
LOCAL_MODEL_DIR = LOCAL_SAVE_BASE / str(TEST_SET) / 'GATraj'
DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/GATraj/colab_savedata') / str(TEST_SET) / 'GATraj'

In [ ]:
# Update the repository without discarding local edits.
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR, check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Local training and resume directory:', LOCAL_MODEL_DIR)
print('Drive backup directory:', DRIVE_MODEL_DIR)

## Check checkpoint saving and loading before training
This runs the repository's actual save/load methods with a tiny CPU model. It checks model weights, optimizer and scheduler restoration, RNG state, and the ZIP file format without running a training epoch on the GPU.


In [ ]:
# Run in a fresh process so the check always imports the updated repository code.
checkpoint_check = r"""
import os
import tempfile
import zipfile
from types import SimpleNamespace
import numpy as np
import torch
from Processor import Processor

with tempfile.TemporaryDirectory() as root:
    os.makedirs(os.path.join(root, 'GATraj'))
    args = SimpleNamespace(save_dir=root, train_model='GATraj', seed=1, load_model=1)
    def make_processor():
        p = Processor.__new__(Processor)
        p.args = args
        p.net = torch.nn.Linear(2, 2)
        p.optimizer = torch.optim.Adam(p.net.parameters(), lr=5e-4)
        p.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(p.optimizer, T_max=1000, eta_min=1e-5)
        return p
    original = make_processor()
    original.net(torch.ones(1, 2)).sum().backward()
    original.optimizer.step()
    original.scheduler.step()
    original.save_model(1)
    path = os.path.join(root, 'GATraj', 'GATraj_1.tar')
    assert zipfile.is_zipfile(path), 'Checkpoint is not in ZIP format; update the repository.'
    saved_rng = torch.load(path, map_location='cpu', weights_only=False)['rng_state']
    restored = make_processor()
    restored.load_model()
    assert restored.epoch == 2
    for name, value in original.net.state_dict().items():
        assert torch.equal(value, restored.net.state_dict()[name])
    assert restored.scheduler.state_dict() == original.scheduler.state_dict()
    for key, state in original.optimizer.state_dict()['state'].items():
        for name, value in state.items():
            assert torch.equal(value, restored.optimizer.state_dict()['state'][key][name])
    assert torch.equal(torch.get_rng_state(), saved_rng['torch'])
    assert np.array_equal(np.random.get_state()[1], saved_rng['numpy'][1].numpy())
print('PASS: CPU checkpoint save/load round trip, optimizer, scheduler, RNG and ZIP format.')
"""
subprocess.run([sys.executable, '-c', checkpoint_check], cwd=REPO_DIR, check=True)

## Train or resume
Keep `TOTAL_EPOCHS = 1000` throughout the run. Rerun the next cell to continue from the newest local checkpoint. An unreadable checkpoint stops the cell before training rather than silently starting over.

The first segment trains epochs 1–100; later segments train 101–200, and so on. If the latest local checkpoint is epoch 437, the next segment runs 438–500. Local checkpoints are saved every epoch; Drive backups are made at the segment boundaries. If a segment fails, its output is printed and the next segment is not started.


In [ ]:
import shutil
import re

assert TOTAL_EPOCHS > 0 and BACKUP_EVERY > 0

def checkpoint_epoch(path):
    match = re.fullmatch(r'GATraj_(\d+)\.tar', path.name)
    return int(match.group(1)) if match else -1

def validate_checkpoint(path, epoch):
    # Only load checkpoints produced by your own training or a trusted source.
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    if int(checkpoint['epoch']) != epoch:
        raise RuntimeError(f'Checkpoint epoch does not match its filename: {path}')
    for key in ('state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'rng_state'):
        if key not in checkpoint:
            raise RuntimeError(f'{path} lacks {key}; use a checkpoint from the updated training code.')
    del checkpoint

def run_command(command):
    # Stream stdout and stderr together so the actual training error is visible.
    with subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'train.py exited with code {returncode}; see its traceback above.')

common_args = [
    '--test_set', str(TEST_SET),
    '--save_base_dir', str(LOCAL_SAVE_BASE) + '/',
    '--train_model', 'GATraj',
    '--num_epochs', str(TOTAL_EPOCHS),
    '--x_encoder_layers', '3', '--eta_min', '1e-5',
    '--batch_size', '32', '--learning_rate', '5e-4',
    '--randomRotate', 'True', '--final_mode', '20',
    '--neighbor_thred', '10', '--using_cuda', 'True', '--gpu', '0',
    '--clip', '1', '--pass_time', '2', '--ifGaussian', 'False',
    '--SR', 'True', '--input_offset', 'True', '--seed', str(SEED),
]

candidates = [p for p in LOCAL_MODEL_DIR.glob('GATraj_*.tar')
              if 0 < checkpoint_epoch(p) <= TOTAL_EPOCHS]
latest_epoch = max((checkpoint_epoch(p) for p in candidates), default=0)
if latest_epoch:
    validate_checkpoint(LOCAL_MODEL_DIR / f'GATraj_{latest_epoch}.tar', latest_epoch)
    print(f'Resuming local checkpoint at epoch {latest_epoch}.')
else:
    print('No local run checkpoint found. Starting from epoch 1.')

# Back up an existing completed run too, without retraining it.
if latest_epoch == TOTAL_EPOCHS:
    source = LOCAL_MODEL_DIR / f'GATraj_{latest_epoch}.tar'
    pending = DRIVE_MODEL_DIR / (source.name + '.tmp')
    shutil.copy2(source, pending)
    pending.replace(DRIVE_MODEL_DIR / source.name)
    print('Training already complete; final checkpoint backed up to Drive.')

while latest_epoch < TOTAL_EPOCHS:
    end_epoch = min((latest_epoch // BACKUP_EVERY + 1) * BACKUP_EVERY, TOTAL_EPOCHS)
    print(f'Training epochs {latest_epoch + 1} through {end_epoch}...')
    run_command([
        sys.executable, '-u', 'train.py', *common_args,
        '--phase', 'train', '--load_model', str(latest_epoch),
        '--epochs_per_run', str(end_epoch - latest_epoch),
    ])
    source = LOCAL_MODEL_DIR / f'GATraj_{end_epoch}.tar'
    validate_checkpoint(source, end_epoch)
    # Write the backup to a temporary name before publishing its final name.
    pending = DRIVE_MODEL_DIR / (source.name + '.tmp')
    shutil.copy2(source, pending)
    pending.replace(DRIVE_MODEL_DIR / source.name)
    print(f'Backed up epoch {end_epoch}: {DRIVE_MODEL_DIR / source.name}')
    latest_epoch = end_epoch

## Evaluate the completed local checkpoint
Evaluation uses the same fold and model arguments as training. It does not load anything from Drive.


In [ ]:
EVAL_EPOCH = TOTAL_EPOCHS
validate_checkpoint(LOCAL_MODEL_DIR / f'GATraj_{EVAL_EPOCH}.tar', EVAL_EPOCH)
run_command([
    sys.executable, '-u', 'train.py', *common_args,
    '--phase', 'test', '--load_model', str(EVAL_EPOCH),
])